# 01 - Data Understanding
**Project:** Predictive Maintenance & Equipment Failure Intelligence System — **Level 1 (Foundations)**

## 1. Project introduction and business problem
A manufacturing company runs machines that produce sensor readings (temperature, speed, torque, tool wear).
Unexpected failures stop production, raise repair costs and create safety risks.
In Level 1 we **understand, clean, explore and test** the data and prepare a leakage-safe dataset for machine learning in Level 2.

## 2. What is predictive maintenance?
| Strategy | Idea | Problem |
|---|---|---|
| Reactive | Repair after it breaks | Costly downtime |
| Preventive | Service on a fixed schedule | Replaces healthy parts |
| **Predictive** | Use data to estimate when failure is likely | Needs good data and models |

Predictive maintenance lets the maintenance team act **before** a failure, only when needed.

## 3. Dataset source and data dictionary
- **Dataset:** AI4I 2020 Predictive Maintenance Dataset (synthetic data that reflects real industrial predictive maintenance)
- **Source:** UCI Machine Learning Repository — https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset
- **File:** `ai4i2020.csv` (placed in `data/raw/`)
- **Licence:** Creative Commons Attribution 4.0 (CC BY 4.0) — free to use with credit to the authors (S. Matzka, 2020).
- **Main target:** `Machine failure` (1 = failed, 0 = did not fail)

### How to download
1. **Automatic:** the cell below downloads the ZIP from UCI if `data/raw/ai4i2020.csv` is missing.
2. **Manual:** open the UCI link above → click **Download** → unzip → copy `ai4i2020.csv` into `data/raw/`.
3. **Google Colab:** upload the project folder (or the CSV) using the Files panel, then run the notebooks from the `notebooks/` folder.

| Column | Meaning | Unit |
|---|---|---|
| UDI | Unique row identifier (1–10000) | – |
| Product ID | Product serial number; first letter = quality type | – |
| Type | Product quality variant: L (low), M (medium), H (high) quality | – |
| Air temperature [K] | Ambient air temperature | Kelvin |
| Process temperature [K] | Process temperature | Kelvin |
| Rotational speed [rpm] | Spindle speed | rev/min |
| Torque [Nm] | Torque | Newton-metre |
| Tool wear [min] | Minutes the tool has been used | minutes |
| Machine failure | Target: 1 if any failure happened | 0/1 |
| TWF | Tool wear failure | 0/1 |
| HDF | Heat dissipation failure | 0/1 |
| PWF | Power failure | 0/1 |
| OSF | Overstrain failure | 0/1 |
| RNF | Random failure | 0/1 |

## 4. Import libraries

In [1]:
# Import the libraries used in this notebook
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sys.path.append("..")          # lets us import our own src/ package
from src import preprocessing as pp

np.random.seed(42)             # fixed seed for reproducibility
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 50)

## 5. Load the dataset (download automatically if needed)

In [2]:
from pathlib import Path
import urllib.request, zipfile, io

RAW_PATH = Path("../data/raw/ai4i2020.csv")
URL = "https://archive.ics.uci.edu/static/public/601/ai4i+2020+predictive+maintenance+dataset.zip"

if not RAW_PATH.exists():
    print("Downloading dataset from UCI ...")
    RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    data = urllib.request.urlopen(URL).read()
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        z.extract("ai4i2020.csv", RAW_PATH.parent)

df = pp.load_raw(RAW_PATH)
print("Loaded:", RAW_PATH, "| shape:", df.shape)

Loaded: ..\data\raw\ai4i2020.csv | shape: (10000, 14)


## 6. First and last five rows

In [3]:
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [4]:
df.tail()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
9995,9996,M24855,M,298.8,308.4,1604,29.5,14,0,0,0,0,0,0
9996,9997,H39410,H,298.9,308.4,1632,31.8,17,0,0,0,0,0,0
9997,9998,M24857,M,299.0,308.6,1645,33.4,22,0,0,0,0,0,0
9998,9999,H39412,H,299.0,308.7,1408,48.5,25,0,0,0,0,0,0
9999,10000,M24859,M,299.0,308.7,1500,40.2,30,0,0,0,0,0,0


## 7. Shape, columns, data types and summary statistics

In [5]:
print("Rows:", df.shape[0], "| Columns:", df.shape[1])
print("\nColumns:", list(df.columns))
print("\nData types:")
print(df.dtypes)

Rows: 10000 | Columns: 14

Columns: ['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']

Data types:
UDI                          int64
Product ID                     str
Type                           str
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Machine failure              int64
TWF                          int64
HDF                          int64
PWF                          int64
OSF                          int64
RNF                          int64
dtype: object


In [6]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
UDI,10000.0,5000.50,2886.90,1.0,2500.75,5000.5,7500.25,10000.0
Air temperature [K],10000.0,300.00,2.00,295.3,298.30,300.1,301.50,304.5
Process temperature [K],10000.0,310.01,1.48,305.7,308.80,310.1,311.10,313.8
Rotational speed [rpm],10000.0,1538.78,179.28,1168.0,1423.00,1503.0,1612.00,2886.0
Torque [Nm],10000.0,39.99,9.97,3.8,33.20,40.1,46.80,76.6
Tool wear [min],10000.0,107.95,63.65,0.0,53.00,108.0,162.00,253.0
Machine failure,10000.0,0.03,0.18,0.0,0.00,0.0,0.00,1.0
TWF,10000.0,0.00,0.07,0.0,0.00,0.0,0.00,1.0
HDF,10000.0,0.01,0.11,0.0,0.00,0.0,0.00,1.0
PWF,10000.0,0.01,0.10,0.0,0.00,0.0,0.00,1.0


## 8. Missing-value analysis

In [7]:
missing = df.isna().sum()
print(missing)
print("\nTotal missing values:", missing.sum())

UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

Total missing values: 0


## 9. Duplicate-record analysis

In [8]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate UDI values:", df["UDI"].duplicated().sum())
print("Duplicate Product IDs:", df["Product ID"].duplicated().sum())
print("Duplicate rows ignoring identifiers:", df.drop(columns=pp.ID_COLS).duplicated().sum())

Exact duplicate rows: 0
Duplicate UDI values: 0
Duplicate Product IDs: 0
Duplicate rows ignoring identifiers: 0


**Note:** rows that are identical once the IDs are removed are *not* errors — different products can legitimately have the same readings. They are kept.

## 10. Unique values and category distributions

In [9]:
print(df.nunique())
print("\nType distribution:")
print(pd.concat([df["Type"].value_counts(), (df["Type"].value_counts(normalize=True)*100).round(1)],
                axis=1, keys=["count", "%"]))
# Does the first letter of Product ID always match Type?
print("\nProduct ID prefix matches Type:", (df["Product ID"].str[0] == df["Type"]).all())

UDI                        10000
Product ID                 10000
Type                           3
Air temperature [K]           93
Process temperature [K]       82
Rotational speed [rpm]       941
Torque [Nm]                  577
Tool wear [min]              246
Machine failure                2
TWF                            2
HDF                            2
PWF                            2
OSF                            2
RNF                            2
dtype: int64

Type distribution:
      count     %
Type             
L      6000  60.0
M      2997  30.0
H      1003  10.0

Product ID prefix matches Type: True


## 11. Target variable and sensor features
- **Target:** `Machine failure`
- **Sensor / process features:** air temperature, process temperature, rotational speed, torque, tool wear
- **Categorical feature:** `Type`
- **Identifiers (not features):** `UDI`, `Product ID`
- **Failure-mode labels:** `TWF, HDF, PWF, OSF, RNF` — they explain *why* a failure happened, so they are only known **after** the event (leakage risk).

In [10]:
print("Target:", pp.TARGET_COL)
print("Sensors:", pp.SENSOR_COLS)
print("Failure modes:", pp.FAILURE_MODE_COLS)

Target: Machine failure
Sensors: ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
Failure modes: ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']


## 12. Failure and non-failure counts and percentages

In [11]:
counts = df[pp.TARGET_COL].value_counts().rename({0: "No failure", 1: "Failure"})
pct = (counts / len(df) * 100).round(2)
print(pd.concat([counts, pct], axis=1, keys=["count", "%"]))

print("\nFailure-mode counts:")
print(df[pp.FAILURE_MODE_COLS].sum())

# Consistency check between failure modes and the main target
any_mode = df[pp.FAILURE_MODE_COLS].sum(axis=1) > 0
print("\nCross-check (rows = Machine failure, cols = any failure mode flagged):")
print(pd.crosstab(df[pp.TARGET_COL], any_mode))
print("Rows with more than one failure mode:", (df[pp.FAILURE_MODE_COLS].sum(axis=1) > 1).sum())

                 count      %
Machine failure              
No failure        9661  96.61
Failure            339   3.39

Failure-mode counts:
TWF     46
HDF    115
PWF     95
OSF     98
RNF     19
dtype: int64

Cross-check (rows = Machine failure, cols = any failure mode flagged):
col_0            False  True 
Machine failure              
0                 9643     18
1                    9    330
Rows with more than one failure mode: 24


**Observation:** the classes are **highly imbalanced** — only about 3.4% of rows are failures.
There are small inconsistencies in the original data: some rows have `Machine failure = 1` with no failure mode,
and some rows have `RNF = 1` while `Machine failure = 0`. These are documented, not "fixed", because we cannot know the truth.

## 13. Possible invalid or physically implausible readings

In [12]:
print(pp.check_implausible(df))
print("\nRows with Tool wear = 0 (new tool, valid):", (df["Tool wear [min]"] == 0).sum())
print("Process temperature lower than air temperature:", (df["Process temperature [K]"] < df["Air temperature [K]"]).sum())

                    column  min_allowed  max_allowed  n_outside
0      Air temperature [K]          250          350          0
1  Process temperature [K]          250          400          0
2   Rotational speed [rpm]            0        10000          0
3              Torque [Nm]            0          200          0
4          Tool wear [min]            0         1000          0

Rows with Tool wear = 0 (new tool, valid): 120
Process temperature lower than air temperature: 0


No readings fall outside the conservative physical limits (e.g. no negative speeds or torques, no impossible temperatures).
Tool wear = 0 simply means a brand-new tool.

## 14. Time sequences or snapshots?
The file has **no timestamp** column and each `Product ID` appears only once.
`UDI` is just a row number. Therefore each row is an **independent snapshot of one product/process**, not a time series of one machine.
➡ Lag, rolling-window or "failures in the last N hours" features cannot be created honestly.

## Save the sensor data dictionary

In [13]:
meaning = {
    "UDI": ("Unique row identifier", "-", "Identifier (exclude)"),
    "Product ID": ("Product serial number; first letter = quality type", "-", "Identifier (exclude)"),
    "Type": ("Product quality variant L/M/H", "-", "Categorical feature"),
    "Air temperature [K]": ("Ambient air temperature", "K", "Sensor feature"),
    "Process temperature [K]": ("Process temperature", "K", "Sensor feature"),
    "Rotational speed [rpm]": ("Spindle rotational speed", "rpm", "Sensor feature"),
    "Torque [Nm]": ("Torque on the tool", "Nm", "Sensor feature"),
    "Tool wear [min]": ("Cumulative tool usage time", "min", "Sensor feature"),
    "Machine failure": ("1 if machine failed", "0/1", "Target"),
    "TWF": ("Tool wear failure", "0/1", "Failure mode (post-event, exclude)"),
    "HDF": ("Heat dissipation failure", "0/1", "Failure mode (post-event, exclude)"),
    "PWF": ("Power failure", "0/1", "Failure mode (post-event, exclude)"),
    "OSF": ("Overstrain failure", "0/1", "Failure mode (post-event, exclude)"),
    "RNF": ("Random failure", "0/1", "Failure mode (post-event, exclude)"),
}
dd = pd.DataFrame([{"column": c, "meaning": meaning[c][0], "dtype": str(df[c].dtype),
                    "unit": meaning[c][1], "missing_values": int(df[c].isna().sum()),
                    "unique_values": int(df[c].nunique()), "role": meaning[c][2]} for c in df.columns])
Path("../reports").mkdir(exist_ok=True)
dd.to_csv("../reports/sensor_data_dictionary.csv", index=False)
dd

,column,meaning,dtype,unit,missing_values,unique_values,role
0,UDI,Unique row identifier,int64,-,0,10000,Identifier (exclude)
1,Product ID,Product serial number; first letter = quality ...,str,-,0,10000,Identifier (exclude)
2,Type,Product quality variant L/M/H,str,-,0,3,Categorical feature
3,Air temperature [K],Ambient air temperature,float64,K,0,93,Sensor feature
4,Process temperature [K],Process temperature,float64,K,0,82,Sensor feature
5,Rotational speed [rpm],Spindle rotational speed,int64,rpm,0,941,Sensor feature
6,Torque [Nm],Torque on the tool,float64,Nm,0,577,Sensor feature
7,Tool wear [min],Cumulative tool usage time,int64,min,0,246,Sensor feature
8,Machine failure,1 if machine failed,int64,0/1,0,2,Target
9,TWF,Tool wear failure,int64,0/1,0,2,"Failure mode (post-event, exclude)"


## 15. Findings summary
- 10,000 rows × 14 columns; **no missing values** and **no duplicate rows**.
- 5 numeric process variables, 1 categorical product type (L 60%, M 30%, H 10%).
- Target `Machine failure`: **339 failures (3.39%)** → strongly imbalanced.
- 5 failure-mode columns are post-event labels and must not be used as predictors.
- Small label inconsistencies exist in the original data (documented, kept).
- Each row is an independent snapshot — no real time order.